# 从 Composition 独立构造特征，同时保留原始数据全部列

这个 Notebook 会**完整读取原始数据表并保留所有原有列**，但新特征的计算只使用 `Composition` 一列。

流程：
1. 完整读取原始 Excel/CSV；
2. 仅用 `Composition` 自动解析 A/B 位、氧化态和混合价；
3. 计算全部组成特征；
4. 将新特征追加到原始数据右侧；
5. 如果原始表已经存在与新特征同名的列，会将原始列改名为 `original_<列名>`，避免原始数据被覆盖。

离子势定义：
\[
IP=\frac{Z^2}{r}
\]

只保留以下 4 个离子势特征：
- `A_z2_over_r_mean`
- `B_z2_over_r_mean`
- `A_z2_over_r_range`
- `B_z2_over_r_range`


In [1]:
import re, math, json
from collections import OrderedDict
import numpy as np

R_O = 1.35
CHI_O_PAULING = 3.44

# Standard atomic weights used by the original feature set.
ATOMIC_WEIGHT = {
'Ag':107.8682,'Al':26.9815385,'As':74.921595,'Ba':137.327,'Bi':208.98040,'Ca':40.078,'Ce':140.116,
'Co':58.933194,'Cr':51.9961,'Cs':132.90545196,'Cu':63.546,'Dy':162.500,'Er':167.259,'Eu':151.964,
'Fe':55.845,'Ga':69.723,'Gd':157.25,'Ge':72.630,'Hf':178.49,'Ho':164.93033,'In':114.818,'Ir':192.217,
'K':39.0983,'La':138.90547,'Li':6.94,'Lu':174.9668,'Mg':24.305,'Mn':54.938044,'Mo':95.95,'Na':22.98976928,
'Nb':92.90637,'Nd':144.242,'Ni':58.6934,'P':30.973761998,'Pb':207.2,'Pd':106.42,'Pr':140.90766,'Rb':85.4678,
'Rh':102.90550,'Ru':101.07,'Sb':121.760,'Sc':44.955908,'Si':28.085,'Sm':150.36,'Sn':118.710,'Sr':87.62,
'Ta':180.94788,'Tb':158.92535,'Te':127.60,'Ti':47.867,'Tm':168.93422,'V':50.9415,'W':183.84,'Y':88.90584,
'Yb':173.045,'Zn':65.38,'Zr':91.224
}

PAULING_X = {
'Ag':1.93,'Al':1.61,'As':2.18,'Ba':0.89,'Bi':2.02,'Ca':1.00,'Ce':1.12,'Co':1.88,'Cr':1.66,'Cs':0.79,
'Cu':1.90,'Dy':1.22,'Er':1.24,'Eu':1.20,'Fe':1.83,'Ga':1.81,'Gd':1.20,'Ge':2.01,'Hf':1.30,'Ho':1.23,
'In':1.78,'Ir':2.20,'K':0.82,'La':1.10,'Li':0.98,'Lu':1.27,'Mg':1.31,'Mn':1.55,'Mo':2.16,'Na':0.93,
'Nb':1.60,'Nd':1.14,'Ni':1.91,'P':2.19,'Pb':2.33,'Pd':2.20,'Pr':1.13,'Rb':0.82,'Rh':2.28,'Ru':2.20,
'Sb':2.05,'Sc':1.36,'Si':1.90,'Sm':1.17,'Sn':1.96,'Sr':0.95,'Ta':1.50,'Tb':1.20,'Te':2.10,'Ti':1.54,
'Tm':1.25,'V':1.63,'W':2.36,'Y':1.22,'Yb':1.10,'Zn':1.65,'Zr':1.33
}

D_GROUP = {'Sc':3,'Ti':4,'V':5,'Cr':6,'Mn':7,'Fe':8,'Co':9,'Ni':10,'Cu':11,'Y':3,'Zr':4,'Nb':5,'Mo':6,
'Ru':8,'Rh':9,'Pd':10,'Ag':11,'Hf':4,'Ta':5,'W':6,'Re':7,'Os':8,'Ir':9,'Pt':10,'Au':11}

# (site, element, oxidation_state, radius[A], ionic electronegativity, nth ionization energy[eV], Lewis acid strength)
DESCRIPTOR_ROWS = [('A', 'Ag', 1, 1.28, 1.93, 7.576234, 0.191), ('A', 'Ba', 2, 1.61, 1.087, 10.0, 0.194), ('A', 'Bi', 3, 1.17, 1.399, 25.56, 0.436), ('A', 'Ca', 2, 1.34, 1.092, 11.87, 0.264), ('A', 'Ce', 3, 1.34, 1.24, 20.2, 0.32), ('A', 'Cs', 1, 1.88, 0.984, 3.89, 0.084), ('A', 'Dy', 3, 1.24, 1.293, 22.8, 0.396), ('A', 'Er', 3, 1.22, 1.298, 22.74, 0.392), ('A', 'Eu', 3, 1.28, 1.299, 24.92, 0.371), ('A', 'Gd', 3, 1.27, 1.263, 20.63, 0.371), ('A', 'Ho', 3, 1.23, 1.296, 22.84, 0.403), ('A', 'K', 1, 1.64, 0.978, 4.34, 0.108), ('A', 'La', 3, 1.36, 1.225, 19.18, 0.343), ('A', 'Li', 1, 1.18, 0.98, 5.391715, 0.215), ('A', 'Lu', 3, 1.19, 1.29, 20.96, 0.423), ('A', 'Mg', 2, 1.03, 1.31, 15.035271, 0.337), ('A', 'Na', 1, 1.39, 0.985, 5.14, 0.159), ('A', 'Nd', 3, 1.31, 1.256, 22.1, 0.363), ('A', 'Pr', 3, 1.32, 1.258, 21.62, 0.326), ('A', 'Rb', 1, 1.72, 0.983, 4.18, 0.099), ('A', 'Sm', 3, 1.28, 1.283, 23.4, 0.38), ('A', 'Sr', 2, 1.44, 1.093, 11.03, 0.222), ('A', 'Tb', 3, 1.25, 1.281, 21.91, 0.375), ('A', 'Tm', 3, 1.21, 1.31, 23.68, 0.413), ('A', 'Y', 3, 1.159, 1.22, 20.52441, 0.393), ('A', 'Yb', 3, 1.2, 1.326, 25.05, 0.416), ('B', 'Ag', 1, 1.15, 1.097, 7.576234, 0.191), ('B', 'Al', 3, 0.535, 1.513, 28.447642, 0.583), ('B', 'As', 3, 0.58, 1.589, 28.349, 0.66), ('B', 'As', 5, 0.46, 2.159, 62.77, 1.235), ('B', 'Bi', 3, 1.03, 1.399, 25.57075, 0.436), ('B', 'Bi', 5, 0.76, 1.895, 54.856, 0.86), ('B', 'Ca', 2, 1.0, 1.16, 11.871719, 0.264), ('B', 'Ce', 3, 1.01, 1.348, 20.2, 0.32), ('B', 'Ce', 4, 0.87, 1.608, 36.76, 0.45), ('B', 'Co', 2, 0.745, 1.321, 17.0844, 0.355), ('B', 'Co', 3, 0.55, 1.69, 33.5, 0.5), ('B', 'Co', 4, 0.53, 2.009, 51.27, 0.666), ('B', 'Cr', 2, 0.8, 1.287, 16.486305, 0.39), ('B', 'Cr', 3, 0.62, 1.59, 30.96, 0.5), ('B', 'Cr', 4, 0.55, 1.883, 49.16, 0.71), ('B', 'Cu', 2, 0.73, 1.37, 20.29, 0.374), ('B', 'Cu', 3, 0.54, 1.8, 36.841, 0.75), ('B', 'Dy', 3, 0.912, 1.426, 22.89, 0.396), ('B', 'Er', 3, 0.89, 1.438, 22.7, 0.392), ('B', 'Fe', 2, 0.78, 1.292, 16.19921, 0.352), ('B', 'Fe', 3, 0.55, 1.65, 30.65, 0.528), ('B', 'Fe', 4, 0.59, 1.912, 54.91, 0.704), ('B', 'Ga', 3, 0.62, 1.579, 30.72576, 0.636), ('B', 'Gd', 3, 0.938, 1.386, 20.625, 0.371), ('B', 'Ge', 4, 0.53, 1.854, 45.7155, 0.928), ('B', 'Hf', 4, 0.71, 1.706, 33.33, 0.59), ('B', 'Ho', 3, 0.901, 1.433, 22.79, 0.403), ('B', 'In', 3, 0.8, 1.81, 28.03, 0.495), ('B', 'Ir', 3, 0.68, 1.649, 28.0, 0.5), ('B', 'Ir', 4, 0.625, 1.881, 40.0, 0.74), ('B', 'Ir', 5, 0.57, 2.183, 57.0, 0.833), ('B', 'La', 3, 1.032, 1.327, 19.1773, 0.343), ('B', 'Lu', 3, 0.861, 1.431, 20.9594, 0.423), ('B', 'Mg', 2, 0.72, 1.234, 15.035271, 0.337), ('B', 'Mn', 2, 0.83, 1.263, 15.63999, 0.334), ('B', 'Mn', 3, 0.58, 1.6275, 33.67, 0.513), ('B', 'Mn', 4, 0.53, 1.912, 51.2, 0.68), ('B', 'Mo', 6, 0.59, 2.101, 68.83, 1.185), ('B', 'Nb', 5, 0.64, 1.86, 50.57, 0.835), ('B', 'Nd', 3, 0.983, 1.382, 22.09, 0.363), ('B', 'Ni', 2, 0.69, 1.367, 18.168838, 0.338), ('B', 'Ni', 3, 0.56, 1.7, 35.19, 0.502), ('B', 'Ni', 4, 0.48, 2.04, 54.92, 0.666), ('B', 'P', 5, 0.38, 2.139, 65.02511, 1.25), ('B', 'Pb', 2, 1.19, 1.225, 15.032499, 0.266), ('B', 'Pb', 4, 0.775, 1.746, 42.33256, 0.72), ('B', 'Pd', 2, 0.86, 1.346, 19.43, 0.5), ('B', 'Pd', 4, 0.615, 1.876, 46.0, 0.666), ('B', 'Rh', 3, 0.665, 1.622, 31.06, 0.5), ('B', 'Rh', 4, 0.6, 1.863, 42.0, 0.666), ('B', 'Ru', 3, 0.68, 1.576, 28.47, 0.5), ('B', 'Ru', 4, 0.62, 1.848, 45.0, 0.666), ('B', 'Ru', 5, 0.565, 2.099, 59.0, 0.833), ('B', 'Sb', 3, 0.76, 1.476, 25.3235, 0.56), ('B', 'Sb', 5, 0.6, 1.971, 56.0, 0.833), ('B', 'Sc', 3, 0.885, 1.42, 24.76, 0.481), ('B', 'Si', 4, 0.4, 1.887, 45.14179, 0.995), ('B', 'Sm', 3, 0.958, 1.41, 23.55, 0.38), ('B', 'Sn', 2, 1.18, 1.181, 14.63307, 0.43), ('B', 'Sn', 4, 0.69, 1.706, 40.74, 0.69), ('B', 'Ta', 5, 0.64, 1.925, 48.272, 0.822), ('B', 'Tb', 3, 0.923, 1.41, 21.82, 0.375), ('B', 'Tb', 4, 0.76, 1.733, 39.33, 0.666), ('B', 'Te', 4, 0.97, 1.467, 37.4155, 0.571), ('B', 'Te', 6, 0.56, 2.18, 69.1, 1.0), ('B', 'Ti', 4, 0.605, 1.73, 43.27, 0.676), ('B', 'Tm', 3, 0.88, 1.455, 23.66, 0.413), ('B', 'V', 5, 0.54, 2.03, 65.28, 1.018), ('B', 'W', 6, 0.6, 2.175, 64.77, 1.033), ('B', 'Y', 3, 0.9, 1.34, 20.52441, 0.393), ('B', 'Yb', 3, 0.868, 1.479, 25.05, 0.416), ('B', 'Zn', 2, 0.74, 1.336, 17.96, 0.405), ('B', 'Zr', 4, 0.72, 1.61, 34.41836, 0.589)]

DESC = {}
for site,el,ox,r,chi,ie,lewis in DESCRIPTOR_ROWS:
    DESC[(site,el,int(ox))] = {'radius':float(r),'electronegativity':float(chi),'ionization_energy':float(ie),'lewis':float(lewis)}
A_ALLOWED = {el for site,el,ox,*_ in DESCRIPTOR_ROWS if site=='A'}
B_ALLOWED = {el for site,el,ox,*_ in DESCRIPTOR_ROWS if site=='B'}
A_STATES = {el:sorted({ox for site,e,ox,*_ in DESCRIPTOR_ROWS if site=='A' and e==el}) for el in A_ALLOWED}
B_STATES = {el:sorted({ox for site,e,ox,*_ in DESCRIPTOR_ROWS if site=='B' and e==el}) for el in B_ALLOWED}

In [2]:
NUM_RE = re.compile(r'(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?')
EL_RE = re.compile(r'[A-Z][a-z]?')
TOKEN_RE = re.compile(r'[A-Z][a-z]?|\(|\)|(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?')

def parse_formula_ordered(formula):
    s=str(formula).strip().replace(' ','').replace('−','-')
    tokens=TOKEN_RE.findall(s)
    if ''.join(tokens)!=s: raise ValueError(f'无法完整解析化学式: {formula}')
    stack=[OrderedDict()]; i=0
    while i<len(tokens):
        t=tokens[i]
        if t=='(':
            stack.append(OrderedDict()); i+=1
        elif t==')':
            if len(stack)==1: raise ValueError(f'括号不匹配: {formula}')
            group=stack.pop(); i+=1; mult=1.0
            if i<len(tokens) and NUM_RE.fullmatch(tokens[i]): mult=float(tokens[i]); i+=1
            for el,v in group.items(): stack[-1][el]=stack[-1].get(el,0.0)+v*mult
        else:
            if not EL_RE.fullmatch(t): raise ValueError(f'非法token {t}: {formula}')
            el=t; i+=1; amt=1.0
            if i<len(tokens) and NUM_RE.fullmatch(tokens[i]): amt=float(tokens[i]); i+=1
            stack[-1][el]=stack[-1].get(el,0.0)+amt
    if len(stack)!=1: raise ValueError(f'括号不匹配: {formula}')
    return stack[0]

def split_ab(formula):
    d=parse_formula_ordered(formula); oxy=float(d.pop('O',0.0)); items=list(d.items())
    if oxy<=0: raise ValueError('化学式中未检测到 O')
    if len(items)<2: raise ValueError('至少需要两类阳离子才能划分 A/B 位')
    candidates=[]
    for cut in range(1,len(items)):
        ai,bi=items[:cut],items[cut:]; aocc=sum(v for _,v in ai); bocc=sum(v for _,v in bi)
        badA=sum(e not in A_ALLOWED for e,_ in ai); badB=sum(e not in B_ALLOWED for e,_ in bi)
        exclusive=sum((e in B_ALLOWED and e not in A_ALLOWED) for e,_ in ai)+sum((e in A_ALLOWED and e not in B_ALLOWED) for e,_ in bi)
        balance=abs(math.log((aocc+1e-12)/(bocc+1e-12)))
        score=1000*(badA+badB)+100*exclusive+balance+1e-6*cut
        candidates.append((score,cut))
    _,cut=min(candidates)
    A=OrderedDict(items[:cut]); B=OrderedDict(items[cut:])
    missingA=[e for e in A if e not in A_ALLOWED]; missingB=[e for e in B if e not in B_ALLOWED]
    if missingA or missingB: raise KeyError(f'描述符数据库缺失/位点无法识别: A缺失={missingA}, B缺失={missingB}')
    return A,B,oxy

def norm(d):
    s=sum(float(v) for v in d.values())
    if s<=0: raise ValueError('位点总占位 <= 0')
    return {k:float(v)/s for k,v in d.items()}

def entropy(frac): return float(-sum(x*math.log(x) for x in frac.values() if x>0))
def wmean(v,w):
    v=np.asarray(v,float);w=np.asarray(w,float);return float(np.sum(v*w)/np.sum(w))
def wstd(v,w):
    m=wmean(v,w);v=np.asarray(v,float);w=np.asarray(w,float);return float(np.sqrt(np.sum(w*(v-m)**2)/np.sum(w)))
def mismatch(v,w):
    m=wmean(v,w);v=np.asarray(v,float);w=np.asarray(w,float);return float(np.sqrt(np.sum(w*(1-v/m)**2)/np.sum(w))) if m else np.nan

def d_count(el,ox): return float(np.clip(D_GROUP[el]-ox,0,10)) if el in D_GROUP else 0.0

def assign_b_states(A,B,oxygen):
    Af,Bf=norm(A),norm(B); acharge=sum(A[e]*A_STATES[e][0] for e in A)
    bocc=sum(B.values()); nominal=(2*oxygen-acharge)/bocc
    qmin=sum(Bf[e]*min(B_STATES[e]) for e in Bf);qmax=sum(Bf[e]*max(B_STATES[e]) for e in Bf)
    lam=0.0 if abs(qmax-qmin)<1e-12 else (nominal-qmin)/(qmax-qmin); lc=min(1.0,max(0.0,lam))
    state_assignment={}; components=[]
    for e,x in Bf.items():
        ss=B_STATES[e]; target=min(ss)+lc*(max(ss)-min(ss))
        if len(ss)==1 or target<=ss[0]+1e-12: dist={ss[0]:1.0}
        elif target>=ss[-1]-1e-12: dist={ss[-1]:1.0}
        else:
            dist=None
            for lo,hi in zip(ss[:-1],ss[1:]):
                if lo-1e-12<=target<=hi+1e-12:
                    if abs(target-lo)<1e-12: dist={lo:1.0}
                    elif abs(target-hi)<1e-12: dist={hi:1.0}
                    else:
                        wh=(target-lo)/(hi-lo);dist={lo:1-wh,hi:wh}
                    break
            if dist is None: raise RuntimeError((e,target,ss))
        state_assignment[e]={f'{z}plus':float(q) for z,q in dist.items() if q>1e-12}
        for z,q in dist.items():
            if q>1e-12: components.append((e,int(z),x*float(q),DESC[('B',e,int(z))]))
    assigned=sum(ox*w for _,ox,w,_ in components)
    warning='' if 0<=lam<=1 else f'电荷守恒要求超出B位可用价态范围: lambda={lam:.8g}, 已截断为{lc:g}; 请核查组成/氧非化学计量或扩展价态表'
    return nominal,assigned,lam,state_assignment,components,warning

def a_components(A):
    Af=norm(A);out=[]
    for e,x in Af.items():
        ss=A_STATES[e]
        if len(ss)!=1: raise ValueError(f'A位 {e} 的价态定义不唯一: {ss}')
        z=ss[0];out.append((e,z,x,DESC[('A',e,z)]))
    return out

In [3]:
def prop_stats(comp,prop):
    vals=[p[prop] for _,_,_,p in comp];w=[x for _,_,x,_ in comp]
    return wmean(vals,w),wstd(vals,w),mismatch(vals,w)

def element_effective_prop(comp,prop):
    d={}
    for e in {r[0] for r in comp}:
        rows=[r for r in comp if r[0]==e]; sw=sum(r[2] for r in rows)
        d[e]=sum(r[2]*r[3][prop] for r in rows)/sw
    return d

def pauling_mean(frac):
    return sum(x*PAULING_X[e] for e,x in frac.items())

def make_features(formula):
    A,B,O=split_ab(formula); Af,Bf=norm(A),norm(B); ac=a_components(A)
    nominal,assigned,lam,state_assignment,bc,warning=assign_b_states(A,B,O)
    ao,bo=sum(A.values()),sum(B.values())
    # base properties
    Ar,Ars,Armm=prop_stats(ac,'radius');Br,Brs,Brmm=prop_stats(bc,'radius')
    Ae,Aes,_=prop_stats(ac,'electronegativity');Be,Bes,_=prop_stats(bc,'electronegativity')
    Ai,Ais,_=prop_stats(ac,'ionization_energy');Bi,Bis,_=prop_stats(bc,'ionization_energy')
    Al,Als,_=prop_stats(ac,'lewis');Bl,Bls,_=prop_stats(bc,'lewis')
    Aeff=element_effective_prop(ac,'electronegativity');Beff=element_effective_prop(bc,'electronegativity')
    dvals=[d_count(e,z) for e,z,_,_ in bc];dw=[w for _,_,w,_ in bc]
    aox=sum(z*w for _,z,w,_ in ac);boxstd=wstd([z for _,z,_,_ in bc],dw)
    ratio=Ar/Br if Br else np.nan;tau=R_O/Br-aox*(aox-ratio/np.log(ratio)) if ratio>0 and not np.isclose(ratio,1) else np.nan
    dA=abs(CHI_O_PAULING-pauling_mean(Af));dB=abs(CHI_O_PAULING-pauling_mean(Bf))
    Aip=[z*z/p['radius'] for _,z,_,p in ac];Aw=[w for _,_,w,_ in ac];Bip=[z*z/p['radius'] for _,z,_,p in bc]
    f={
      'A_site_fraction_2plus':sum(w for _,z,w,_ in ac if z==2),'A_site_fraction_3plus':sum(w for _,z,w,_ in ac if z==3),
      'B_site_fraction_2plus':sum(w for _,z,w,_ in bc if z==2),'B_site_fraction_3plus':sum(w for _,z,w,_ in bc if z==3),'B_site_fraction_4plus':sum(w for _,z,w,_ in bc if z==4),
      'B_site_oxidation':nominal,'tolerance_factor':(Ar+R_O)/(math.sqrt(2)*(Br+R_O)),
      'A_site_Ba_fraction':Af.get('Ba',0.0),'A_site_Sr_fraction':Af.get('Sr',0.0),'B_site_Co_fraction':Bf.get('Co',0.0),'B_site_Fe_fraction':Bf.get('Fe',0.0),
      'composition_entropy':entropy(Af)+entropy(Bf),'A_site_radius':Ar,'B_site_radius':Br,'A_site_electronegativity':Ae,'B_site_electronegativity':Be,
      'A_site_first_ionization_energy':Ai,'B_site_first_ionization_energy':Bi,'A_site_Lewis_acid_strength':Al,'B_site_Lewis_acid_strength':Bl,
      'B_site_d':wmean(dvals,dw),'A_atomic_weight':sum(Af[e]*ATOMIC_WEIGHT[e] for e in Af),'B_atomic_weight':sum(Bf[e]*ATOMIC_WEIGHT[e] for e in Bf),
      'A_site_electronegativity_difference':max(Aeff.values())-min(Aeff.values()),'B_site_electronegativity_difference':max(Beff.values())-min(Beff.values()),
      'oxygen_per_B':O/bo,'nominal_oxygen_deficiency':3-O/bo,'A_B_ratio':ao/bo,
      'A_num_elements':len(Af),'B_num_elements':len(Bf),'A_entropy':entropy(Af),'B_entropy':entropy(Bf),'A_max_fraction':max(Af.values()),'B_max_fraction':max(Bf.values()),
      'A_radius_std':Ars,'B_radius_std':Brs,'A_radius_mismatch':Armm,'B_radius_mismatch':Brmm,
      'A_electronegativity_std':Aes,'B_electronegativity_std':Bes,'A_ionization_energy_std':Ais,'B_ionization_energy_std':Bis,'A_Lewis_std':Als,'B_Lewis_std':Bls,
      'A_site_fraction_1plus':sum(w for _,z,w,_ in ac if z==1),'A_site_oxidation':aox,'B_site_fraction_1plus':sum(w for _,z,w,_ in bc if z==1),'B_site_fraction_5plus':sum(w for _,z,w,_ in bc if z==5),'B_site_fraction_6plus':sum(w for _,z,w,_ in bc if z==6),'B_oxidation_std':boxstd,
      'octahedral_factor':Br/R_O,'A_O_radius_ratio':Ar/R_O,'A_B_radius_ratio':ratio,'bartel_tolerance_factor':tau,
      'A_O_Pauling_electronegativity_difference':dA,'B_O_Pauling_electronegativity_difference':dB,'B_O_Pauling_ionicity':1-math.exp(-(dB**2)/4),
      'B_d_std':wstd(dvals,dw),'B_d_range':float(max(dvals)-min(dvals)),
      'A_z2_over_r_mean':wmean(Aip,Aw),
      'B_z2_over_r_mean':wmean(Bip,dw),
      'A_z2_over_r_range':float(max(Aip)-min(Aip)),
      'B_z2_over_r_range':float(max(Bip)-min(Bip))
    }
    audit={'A_formula':dict(A),'B_formula':dict(B),'oxygen_amount':O,'A_site_occupancy':ao,'B_site_occupancy':bo,'nominal_B_oxidation':nominal,'descriptor_assigned_B_oxidation':assigned,'lambda':lam,'B_state_assignment':state_assignment,'warning':warning}
    return f,audit

In [4]:
FEATURE_DEFINITIONS = {
    "A_site_fraction_2plus":"A位+2价离子比例（A位归一化）","A_site_fraction_3plus":"A位+3价离子比例（A位归一化）",
    "B_site_fraction_2plus":"B位+2价离子比例（混合价加权）","B_site_fraction_3plus":"B位+3价离子比例（混合价加权）","B_site_fraction_4plus":"B位+4价离子比例（混合价加权）",
    "B_site_oxidation":"由化学式计量和电荷守恒得到的名义B位平均氧化态","tolerance_factor":"Goldschmidt容忍因子=(rA+rO)/(sqrt(2)*(rB+rO)), rO=1.35 Å",
    "A_site_Ba_fraction":"A位Ba比例","A_site_Sr_fraction":"A位Sr比例","B_site_Co_fraction":"B位Co比例","B_site_Fe_fraction":"B位Fe比例",
    "composition_entropy":"A/B位分别归一化后，除O元素的 -Σxlnx 之和","A_site_radius":"A位平均离子半径/Å","B_site_radius":"B位平均离子半径/Å",
    "A_site_electronegativity":"A位平均氧化态相关电负性","B_site_electronegativity":"B位平均氧化态相关电负性",
    "A_site_first_ionization_energy":"A位平均对应氧化态的第n电离能/eV","B_site_first_ionization_energy":"B位平均对应氧化态的第n电离能/eV",
    "A_site_Lewis_acid_strength":"A位平均Lewis酸强度","B_site_Lewis_acid_strength":"B位平均Lewis酸强度","B_site_d":"B位平均离子d电子数；非过渡金属=0",
    "A_atomic_weight":"A位平均原子质量","B_atomic_weight":"B位平均原子质量","A_site_electronegativity_difference":"A位各元素有效电负性max-min；单元素=0","B_site_electronegativity_difference":"B位各元素有效电负性max-min；单元素=0",
    "oxygen_per_B":"名义O/B化学计量比","nominal_oxygen_deficiency":"按ABO3参考定义的名义氧缺位代理 3-O/B；不是实验氧空位浓度","A_B_ratio":"A位总占位/B位总占位",
    "A_num_elements":"A位元素种类数","B_num_elements":"B位元素种类数","A_entropy":"A位组态熵 -Σxlnx","B_entropy":"B位组态熵 -Σxlnx","A_max_fraction":"A位最大元素摩尔分数","B_max_fraction":"B位最大元素摩尔分数",
    "A_radius_std":"A位离子半径加权标准差/Å","B_radius_std":"B位元素-价态离子半径加权标准差/Å","A_radius_mismatch":"A位相对半径失配 sqrt[Σx(1-r/rmean)^2]","B_radius_mismatch":"B位相对半径失配 sqrt[Σx(1-r/rmean)^2]",
    "A_electronegativity_std":"A位电负性加权标准差","B_electronegativity_std":"B位元素-价态电负性加权标准差","A_ionization_energy_std":"A位电离能加权标准差/eV","B_ionization_energy_std":"B位元素-价态电离能加权标准差/eV","A_Lewis_std":"A位Lewis酸强度加权标准差","B_Lewis_std":"B位元素-价态Lewis酸强度加权标准差",
    "A_site_fraction_1plus":"A位+1价离子比例","A_site_oxidation":"A位平均氧化态","B_site_fraction_1plus":"B位+1价离子比例","B_site_fraction_5plus":"B位+5价离子比例","B_site_fraction_6plus":"B位+6价离子比例","B_oxidation_std":"B位价态加权标准差",
    "octahedral_factor":"八面体因子 rB/rO，rO=1.35 Å","A_O_radius_ratio":"rA/rO","A_B_radius_ratio":"rA/rB","bartel_tolerance_factor":"Bartel τ = rO/rB - nA[nA-(rA/rB)/ln(rA/rB)]",
    "A_O_Pauling_electronegativity_difference":"|χO-χA|，中性原子Pauling标度，χO=3.44","B_O_Pauling_electronegativity_difference":"|χO-χB|，中性原子Pauling标度，χO=3.44","B_O_Pauling_ionicity":"Pauling离子性代理 1-exp[-(ΔχB-O)^2/4]",
    "B_d_std":"B位离子d电子数加权标准差","B_d_range":"B位离子d电子数max-min",
    "A_z2_over_r_mean":"A位离子势 Z²/r 的组成-价态加权平均",
    "B_z2_over_r_mean":"B位离子势 Z²/r 的组成-价态加权平均",
    "A_z2_over_r_range":"A位 Z²/r 极差 = max(Z²/r)-min(Z²/r)",
    "B_z2_over_r_range":"B位 Z²/r 极差 = max(Z²/r)-min(Z²/r)"
}


In [5]:
# ========================= 只需要修改这里 =========================
from pathlib import Path
import pandas as pd

INPUT = Path(r"D:\Users\lihao\Desktop\SanHuan\ML_code\EACOMP\data\data_923K_2026_09_09_v4.xlsx")  # 支持 .xlsx / .xls / .csv
SHEET_NAME = 0                     # Excel 输入时，数据所在 sheet；0 表示第一张表
COMPOSITION_COL = "Composition"
OUTPUT = INPUT.with_name(INPUT.stem + "_v3.xlsx")
# ================================================================

# 1. 完整读取原始数据：保留所有原始列
if INPUT.suffix.lower() == ".csv":
    raw = pd.read_csv(INPUT)
else:
    raw = pd.read_excel(INPUT, sheet_name=SHEET_NAME)

if COMPOSITION_COL not in raw.columns:
    raise KeyError(f"找不到列: {COMPOSITION_COL}")

raw = raw.copy()
raw[COMPOSITION_COL] = raw[COMPOSITION_COL].astype(str).str.strip()

# 2. 仅使用 Composition 构造特征
feature_rows, audit_rows = [], []

for i, comp in enumerate(raw[COMPOSITION_COL]):
    try:
        f, a = make_features(comp)
        feature_rows.append(f)
        audit_rows.append({
            "row_index": i,
            "Composition": comp,
            "status": "warning" if a["warning"] else "ok",
            "error": "",
            "A_formula": json.dumps(a["A_formula"], ensure_ascii=False),
            "B_formula": json.dumps(a["B_formula"], ensure_ascii=False),
            "oxygen_amount": a["oxygen_amount"],
            "A_site_occupancy": a["A_site_occupancy"],
            "B_site_occupancy": a["B_site_occupancy"],
            "nominal_B_oxidation": a["nominal_B_oxidation"],
            "descriptor_assigned_B_oxidation": a["descriptor_assigned_B_oxidation"],
            "lambda": a["lambda"],
            "B_state_assignment": json.dumps(a["B_state_assignment"], ensure_ascii=False),
            "warning": a["warning"]
        })
    except Exception as e:
        feature_rows.append({})
        audit_rows.append({
            "row_index": i,
            "Composition": comp,
            "status": "error",
            "error": str(e),
            "A_formula": "",
            "B_formula": "",
            "oxygen_amount": np.nan,
            "A_site_occupancy": np.nan,
            "B_site_occupancy": np.nan,
            "nominal_B_oxidation": np.nan,
            "descriptor_assigned_B_oxidation": np.nan,
            "lambda": np.nan,
            "B_state_assignment": "",
            "warning": ""
        })

new_features = pd.DataFrame(feature_rows)

# 3. 如果原表中已经存在与新特征同名的列，保留原值并重命名
rename_map = {}
for col in new_features.columns:
    if col in raw.columns:
        new_name = f"original_{col}"
        k = 1
        while new_name in raw.columns or new_name in rename_map.values():
            new_name = f"original_{col}_{k}"
            k += 1
        rename_map[col] = new_name

if rename_map:
    raw = raw.rename(columns=rename_map)
    print("检测到原表中存在同名特征列，已保留并重命名：")
    for old, new in rename_map.items():
        print(f"  {old} -> {new}")

# 4. 保留全部原始列，并把新生成特征追加到右侧
features = pd.concat(
    [raw.reset_index(drop=True), new_features.reset_index(drop=True)],
    axis=1
)

audit = pd.DataFrame(audit_rows)
definitions = pd.DataFrame({
    "feature": list(FEATURE_DEFINITIONS),
    "definition": list(FEATURE_DEFINITIONS.values())
})
desc_df = pd.DataFrame(
    DESCRIPTOR_ROWS,
    columns=["site","element","oxidation_state","radius",
             "electronegativity","ionization_energy","lewis"]
)

# 5. 导出
with pd.ExcelWriter(OUTPUT) as writer:
    features.to_excel(writer, sheet_name="features", index=False)
    audit.to_excel(writer, sheet_name="audit", index=False)
    definitions.to_excel(writer, sheet_name="feature_definitions", index=False)
    desc_df.to_excel(writer, sheet_name="descriptor_table_builtin", index=False)

print(f"\n完成: {OUTPUT}")
print(f"输入行数: {len(raw)}")
print(f"原始列数: {raw.shape[1]}")
print(f"新增特征数: {new_features.shape[1]}")
print(f"最终总列数: {features.shape[1]}")
print("\n状态统计:")
print(audit["status"].value_counts(dropna=False))

display(features.head())



完成: D:\Users\lihao\Desktop\SanHuan\ML_code\EACOMP\data\data_923K_2026_09_09_v4_v3.xlsx
输入行数: 306
原始列数: 2
新增特征数: 63
最终总列数: 65

状态统计:
status
ok         260
warning     46
Name: count, dtype: int64


,Composition,ASR,A_site_fraction_2plus,A_site_fraction_3plus,B_site_fraction_2plus,B_site_fraction_3plus,B_site_fraction_4plus,B_site_oxidation,tolerance_factor,A_site_Ba_fraction,...,bartel_tolerance_factor,A_O_Pauling_electronegativity_difference,B_O_Pauling_electronegativity_difference,B_O_Pauling_ionicity,B_d_std,B_d_range,A_z2_over_r_mean,B_z2_over_r_mean,A_z2_over_r_range,B_z2_over_r_range
0,SrNb0.175V0.025Co0.8O3,0.004911,1.0,0.0,0.0,0.200,0.6,4.0,1.036421,0.0,...,3.880979,2.49,1.61525,0.479132,2.135416,6.0,2.777778,29.379280,0.0,29.932660
1,SrNb0.15V0.05Co0.8O3,0.004964,1.0,0.0,0.0,0.200,0.6,4.0,1.037784,0.0,...,3.890976,2.49,1.61450,0.478816,2.135416,6.0,2.777778,29.560125,0.0,29.932660
2,SrNb0.1Ta0.1Co0.8O3,0.005855,1.0,0.0,0.0,0.200,0.6,4.0,1.035062,0.0,...,3.871198,2.49,1.62600,0.483649,2.135416,6.0,2.777778,29.198435,0.0,22.698864
3,SrSc0.175Ta0.025Co0.8O3,0.006565,1.0,0.0,0.0,0.175,0.8,4.0,1.014373,0.0,...,3.745737,2.49,1.66050,0.498080,2.000000,5.0,2.777778,26.907167,0.0,28.893008
4,SrNb0.1V0.1Co0.8O3,0.006970,1.0,0.0,0.0,0.200,0.6,4.0,1.040521,0.0,...,3.911625,2.49,1.61300,0.478185,2.135416,6.0,2.777778,29.921814,0.0,29.932660


In [6]:
# 可选检查：确认原始列均保留，并检查 4 个 IP 特征
problem = audit[audit["status"] != "ok"]
print(f"warning/error 行数: {len(problem)}")
if len(problem):
    display(problem[["row_index","Composition","status","error","warning"]].head(30))

ip_cols = [
    "A_z2_over_r_mean",
    "B_z2_over_r_mean",
    "A_z2_over_r_range",
    "B_z2_over_r_range"
]

print("\n4 个离子势特征:")
display(features[[COMPOSITION_COL] + ip_cols].head(10))

print("\n离子势特征缺失值数量:")
print(features[ip_cols].isna().sum())

print("\n输出列数:", features.shape[1])
print("前 20 列:")
print(features.columns[:20].tolist())
print("\n最后 20 列:")
print(features.columns[-20:].tolist())


warning/error 行数: 46


,row_index,Composition,status,error,warning
3,3,SrSc0.175Ta0.025Co0.8O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.09375, 已截断为1; 请核查组成..."
7,7,BaCo0.7Fe0.22Y0.08O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.0434783, 已截断为1; 请核查..."
15,15,BaCo0.7Fe0.22Sc0.08O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.0434783, 已截断为1; 请核查..."
16,16,SrSc0.175Nb0.025Co0.8O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.09375, 已截断为1; 请核查组成..."
18,18,Sr0.95Sc0.175Nb0.025Co0.8O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.15625, 已截断为1; 请核查组成..."
24,24,SrSc0.175V0.025Co0.8O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.09375, 已截断为1; 请核查组成..."
31,31,Ba0.9Co0.7Fe0.2Nb0.1O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.0555556, 已截断为1; 请核查..."
38,38,BaCo0.75Sc0.25O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.1666667, 已截断为1; 请核查..."
43,43,SrSc0.2Co0.8O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.125, 已截断为1; 请核查组成/氧..."
48,48,SrSc0.075Ta0.025Fe0.9O3,warning,,"电荷守恒要求超出B位可用价态范围: lambda=1.0277778, 已截断为1; 请核查..."



4 个离子势特征:


,Composition,A_z2_over_r_mean,B_z2_over_r_mean,A_z2_over_r_range,B_z2_over_r_range
0,SrNb0.175V0.025Co0.8O3,2.777778,29.379280,0.0,29.932660
1,SrNb0.15V0.05Co0.8O3,2.777778,29.560125,0.0,29.932660
2,SrNb0.1Ta0.1Co0.8O3,2.777778,29.198435,0.0,22.698864
3,SrSc0.175Ta0.025Co0.8O3,2.777778,26.907167,0.0,28.893008
4,SrNb0.1V0.1Co0.8O3,2.777778,29.921814,0.0,29.932660
5,SrTa0.15V0.05Co0.8O3,2.777778,29.560125,0.0,29.932660
6,SrTa0.175V0.025Co0.8O3,2.777778,29.379280,0.0,29.932660
7,BaCo0.7Fe0.22Y0.08O3,2.484472,27.898177,0.0,20.188679
8,SrCo0.8Ta0.15V0.05O3,2.777778,29.560125,0.0,29.932660
9,SrCo0.8Ta0.16W0.04O3,2.777778,29.482933,0.0,43.636364



离子势特征缺失值数量:
A_z2_over_r_mean     0
B_z2_over_r_mean     0
A_z2_over_r_range    0
B_z2_over_r_range    0
dtype: int64

输出列数: 65
前 20 列:
['Composition', 'ASR', 'A_site_fraction_2plus', 'A_site_fraction_3plus', 'B_site_fraction_2plus', 'B_site_fraction_3plus', 'B_site_fraction_4plus', 'B_site_oxidation', 'tolerance_factor', 'A_site_Ba_fraction', 'A_site_Sr_fraction', 'B_site_Co_fraction', 'B_site_Fe_fraction', 'composition_entropy', 'A_site_radius', 'B_site_radius', 'A_site_electronegativity', 'B_site_electronegativity', 'A_site_first_ionization_energy', 'B_site_first_ionization_energy']

最后 20 列:
['B_Lewis_std', 'A_site_fraction_1plus', 'A_site_oxidation', 'B_site_fraction_1plus', 'B_site_fraction_5plus', 'B_site_fraction_6plus', 'B_oxidation_std', 'octahedral_factor', 'A_O_radius_ratio', 'A_B_radius_ratio', 'bartel_tolerance_factor', 'A_O_Pauling_electronegativity_difference', 'B_O_Pauling_electronegativity_difference', 'B_O_Pauling_ionicity', 'B_d_std', 'B_d_range', 'A_z2_over_r_mean'